In [ ]:
import numpy as np
from scipy.spatial import KDTree, kdtree
import scipy.spatial.kdtree
import pandas as pd
from sklearn.datasets import load_iris
import matplotlib.pyplot as plt


def knn(df, label, k):
    # build a kd-tree out of input data frame
    # query for each point the nearest neighbors classify
    X = df.drop(columns=[label])
    tree = KDTree(X.to_numpy())
    predicted = []

    for idx, row in df.iterrows():
        _, neighbors = tree.query(X.iloc[idx], k=k)

        values, counts = np.unique(
            df.iloc[neighbors][label].tolist(), return_counts=True
        )

        label_idx = np.argmax(counts)
        predicted_label = values[label_idx]
        predicted.append(predicted_label)

    df["predicted"] = predicted
    return df


def main():
    iris = load_iris()
    feature_names = [
        name.replace(" (cm)", "").replace(" ", "_") for name in iris.feature_names
    ]
    df = pd.DataFrame(iris.data, columns=feature_names)
    df["label"] = iris.target

    k_values = [1, 5, 15]
    for k in k_values:
        result = knn(df.copy(), "label", k=k)
        accuracy = (result["label"] == result["predicted"]).mean()
        print(f"kNN accuracy on iris (k={k}): {accuracy:.4f}")

    feature_x = feature_names[0]
    feature_y = feature_names[1]
    fig, axes = plt.subplots(
        1, len(k_values), figsize=(12, 4), sharex=True, sharey=True
    )
    for ax, k in zip(axes, k_values):
        result = knn(df.copy(), "label", k=k)
        ax.scatter(
            result[feature_x],
            result[feature_y],
            c=result["predicted"],
            cmap="viridis",
            s=18,
            alpha=0.85,
        )
        ax.set_title(f"k={k}")
        ax.set_xlabel(feature_x)
        ax.set_ylabel(feature_y)

    fig.suptitle("kNN predictions on iris (first two features)")
    fig.tight_layout()
    fig.savefig("knn_iris_k_plots.png", dpi=150)


if __name__ == "__main__":
    main()


In [ ]:
from unicodedata import normalize
import numpy as np
import pandas as pd
from pandas._libs.hashtable import value_count
import matplotlib.pyplot as plt


class TreeNode:
    def __init__(self, cond, feature, lvl):
        # condition is a number value which split on
        # left is the true path
        self.condition = cond
        self.condition_feature = feature
        self.level = lvl
        self.prediction = None
        self.left = None
        self.right = None

    def next(self, x_i):
        # test with condition and return the next node approprietly
        return (
            self.left if (x_i[self.condition_feature] < self.condition) else self.right
        )


def find_split_ig(df, label):
    if len(df) == 0:
        return None, None, None
    X = df.drop(columns=[label])
    candidates = []
    for col in X.columns:
        sorted_vals = np.sort(df[col].to_numpy())
        midpoints = [(a + b) / 2 for a, b in zip(sorted_vals, sorted_vals[1:])]
        if not midpoints:
            continue
        ig = []
        for point in midpoints:
            left_mask = df[col] < point
            right_mask = ~left_mask
            p_left = (left_mask).mean()
            p_right = 1.0 - p_left
            p_y_left = df.loc[left_mask, label].value_counts(normalize=True)
            p_y_right = df.loc[right_mask, label].value_counts(normalize=True)
            h_left = -(p_y_left * np.log2(p_y_left)).sum()
            h_right = -(p_y_right * np.log2(p_y_right)).sum()
            h_y_x = p_left * h_left + p_right * h_right
            p_y = (df[label]).value_counts(normalize=True)
            h_y = -(p_y * np.log2(p_y)).sum()
            ig.append(h_y - h_y_x)
        best_idx = int(np.argmax(ig))
        candidates.append((midpoints[best_idx], ig[best_idx], col))
    if not candidates:
        return None, None, None
    return max(candidates, key=lambda item: item[1])


def build_tree(df, label, curr):
    if curr is None or curr.condition is None or curr.condition_feature is None:
        return
    curr.prediction = df[label].mode().iat[0]
    if curr.level == 2:
        return
    else:
        left_subset = df.loc[df[curr.condition_feature] < curr.condition]
        left_cond, _, left_feature = find_split_ig(left_subset, label)
        if left_cond is not None and left_feature is not None:
            curr.left = TreeNode(left_cond, left_feature, curr.level + 1)
            build_tree(left_subset, label, curr.left)

        right_subset = df.loc[df[curr.condition_feature] >= curr.condition]
        right_cond, _, right_feature = find_split_ig(right_subset, label)
        if right_cond is not None and right_feature is not None:
            curr.right = TreeNode(right_cond, right_feature, curr.level + 1)
            build_tree(right_subset, label, curr.right)


def predict_one(node, x_i):
    while node and (node.left or node.right):
        next_node = node.next(x_i)
        if next_node is None:
            break
        node = next_node
    return None if node is None else node.prediction


def mse_report(root, df, label):
    preds = df.apply(lambda row: predict_one(root, row), axis=1).to_numpy()
    y_true = df[label].to_numpy()
    return np.mean((preds - y_true) ** 2)


def print_conditions(node, indent=""):
    if node is None:
        return
    if node.condition is None or node.condition_feature is None:
        print(f"{indent}<no split> (level={node.level}, pred={node.prediction})")
        return
    print(
        f"{indent}{node.condition_feature} < {node.condition:.4f} "
        f"(level={node.level}, pred={node.prediction})"
    )
    if node.left or node.right:
        print_conditions(node.left, indent + "  ")
        print_conditions(node.right, indent + "  ")


def main():
    rng = np.random.default_rng(0)
    n = 200
    x1a = rng.normal(loc=[-1.2, -0.8], scale=0.9, size=(n // 4, 2))
    x1b = rng.normal(loc=[0.8, -1.1], scale=0.9, size=(n // 4, 2))
    x2a = rng.normal(loc=[1.2, 1.0], scale=0.9, size=(n // 4, 2))
    x2b = rng.normal(loc=[-0.9, 1.1], scale=0.9, size=(n // 4, 2))
    X = np.vstack([x1a, x1b, x2a, x2b])
    y = np.array([0] * (n // 2) + [1] * (n // 2))
    noise_idx = rng.choice(n, size=int(0.08 * n), replace=False)
    y[noise_idx] = 1 - y[noise_idx]

    plt.figure(figsize=(5, 4))
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", s=18, alpha=0.85)
    plt.title("Synthetic 2D data (with noise)")
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.tight_layout()
    plt.savefig("decision_tree_data.png", dpi=150)
    df = pd.DataFrame(X, columns=["x1", "x2"])
    df["y"] = y
    label = "y"
    cond, _, feature = find_split_ig(df, label)
    root = TreeNode(cond, feature, 0)
    build_tree(df, label, root)
    print_conditions(root)
    print(f"MSE: {mse_report(root, df, label):.4f}")


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np


def predict(df, label, x):
    df_cols = df.drop(columns=[label]).columns
    p_y = df[label].value_counts(normalize=True)
    y_probs = []
    for y_name in p_y.index.tolist():
        p_xi = []
        for col in df_cols:
            n = len(df.loc[df[label] == y_name])
            p_xi.append(len((df.loc[df[label] == y_name]).loc[df[col] == x[col]]) / n)
        y_probs.append(p_y[y_name] * np.prod(p_xi))
    return p_y.index.tolist()[np.argmax(y_probs)]


def main():
    table_rows = [
        {"spam": "yes", "contains_win": "yes", "contains_free": "yes", "count": 40},
        {"spam": "yes", "contains_win": "yes", "contains_free": "no", "count": 25},
        {"spam": "yes", "contains_win": "no", "contains_free": "yes", "count": 30},
        {"spam": "yes", "contains_win": "no", "contains_free": "no", "count": 5},
        {"spam": "no", "contains_win": "yes", "contains_free": "yes", "count": 5},
        {"spam": "no", "contains_win": "yes", "contains_free": "no", "count": 15},
        {"spam": "no", "contains_win": "no", "contains_free": "yes", "count": 20},
        {"spam": "no", "contains_win": "no", "contains_free": "no", "count": 60},
    ]
    expanded_rows = []
    for row in table_rows:
        expanded_rows.extend(
            {
                "spam": row["spam"],
                "contains_win": row["contains_win"],
                "contains_free": row["contains_free"],
            }
            for _ in range(row["count"])
        )
    df = pd.DataFrame(expanded_rows)
    print(predict(df, "spam", {"contains_win": "no", "contains_free": "no"}))


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression as SkLogisticRegression
import matplotlib.pyplot as plt


class LogisticRegression:
    def __init__(self, df, label, eta=0.01, epochs=1000):
        self.data = df.drop(columns=[label])
        self.label = df[label]
        self.eta = eta
        self.epochs = epochs
        self.b = 0
        self.w = np.zeros(self.data.shape[1], dtype=float)

    def forward(self, x):
        return 1 / (1 + np.exp(-(self.b + np.dot(self.w, x))))

    def log_prob(self, x, y):
        p = self.forward(x)
        grad_b = y - p
        grad_w = x * (y - p)
        return (grad_w, grad_b)

    def mse(self, X=None, y=None):
        if X is None:
            X = self.data.to_numpy()
        if y is None:
            y = self.label.to_numpy()
        preds = np.array([self.forward(x) for x in X])
        return np.mean((preds - y) ** 2)

    def train(self):
        for _ in range(self.epochs):
            (grad_w, grad_b) = map(
                sum,
                zip(
                    *[
                        self.log_prob(x, y)
                        for x, y in zip(self.data.to_numpy(), self.label.to_numpy())
                    ]
                ),
            )
            self.b += self.eta * grad_b
            self.w += self.eta * grad_w


def main():
    iris = load_iris(as_frame=True)
    df = iris.frame
    df = df.loc[df["target"] < 2]
    feature_cols = [iris.feature_names[0], iris.feature_names[2]]
    df = df[feature_cols + ["target"]]

    reg = LogisticRegression(df, "target", eta=0.001, epochs=10000)
    reg.train()
    print(f"MSE: {reg.mse():.4f}")

    w1, w2 = reg.w
    b = reg.b
    print(f"Decision boundary: {w1:.4f} x1 + {w2:.4f} x2 + {b:.4f} = 0")

    x1 = df[feature_cols[0]].to_numpy()
    x2 = df[feature_cols[1]].to_numpy()
    y = df["target"].to_numpy()

    x_min, x_max = x1.min() - 0.5, x1.max() + 0.5
    x_line = np.array([x_min, x_max])
    y_line = -(w1 * x_line + b) / w2 if abs(w2) > 1e-8 else np.full_like(x_line, np.nan)

    plt.figure(figsize=(5, 4))
    plt.scatter(x1, x2, c=y, cmap="viridis", s=18, alpha=0.85)
    plt.plot(x_line, y_line, color="k", lw=1.5)
    plt.title("Logistic regression decision boundary")
    plt.xlabel(feature_cols[0])
    plt.ylabel(feature_cols[1])
    plt.tight_layout()
    plt.savefig("logistic_boundary.png", dpi=150)


if __name__ == "__main__":
    main()
